# 05. 감정·원인 구절 추출 (EOT-DETECT, Colab 실행용, 2026-10-06)

**목적**: 04 노트북에서 뽑은 화장지 리뷰 200건에 Attri et al. (2025, *Findings of EMNLP*)의 **EOT-DETECT** 프롬프트를 적용해, 리뷰마다 **감정(Plutchik 8개 + Neutral)** 과 **원인 구절(Opinion Trigger)** 을 뽑는다.

**논문 방식과 이 노트북**
| 논문 | 이 노트북 |
|---|---|
| 공개 모델 중 **Mistral-7B-Instruct-v0.2 + EOT-DETECT** 가 감정 F1 0.86으로 최고 (Table 2) | 같은 모델 사용 |
| 프롬프트 P = ⟨S, T, I, R⟩: 시스템 메시지, 과제 설명(과제 정의·trigger 정의·감정 범위·지침), 5단계 지시(I1~I5), 리뷰 (4.3절) | 논문이 프롬프트 전문을 공개하지 않아서(GitHub에 LICENSE만 있음, 10/6 확인) **4.3절·부록 D 문구로 재구성** |
| 추론 설정 temperature 0.2, top_p 0.95, top_k 25, max_tokens 2500, n 1 (부록 B) | 그대로 |
| 출력 JSON 형식 (부록 D.8) | 그대로 |
| trigger는 리뷰의 정확한 부분 문자열이어야 함 (3.3절 extractive constraint) | 출력 후 **글자 그대로 있는지 검사**하고 통과한 것만 분석에 씀 |
| H100 GPU, 원래 정밀도 | Colab 무료 T4 GPU라 **4비트 양자화**로 실행 (메모리 부족 때문. 논문과 다른 점으로 보고서에 적는다) |

**실행 환경**: Google Colab. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 고른다.
**걸리는 시간**: 모델 내려받기 몇 분 + 리뷰 200건 생성. 중간에 끊겨도 이어서 돌릴 수 있게 결과를 Google Drive에 한 줄씩 저장한다.
**출력** (Drive의 `MyDrive/cart_to_chart/eot_pilot/`): `eot_raw.jsonl`(모델 원문 출력), `eot_triggers.csv`, `eot_emotions.csv` → 다 끝나면 내려받아 저장소 `data/processed/eot_pilot/`에 넣고 커밋한다.

**[점검]** 표시가 있는 출력을 Claude에게 보내서 확인받는다.

## 0. 처음 한 번만: Hugging Face 준비

Mistral 모델은 Hugging Face에서 **사용 조건에 동의**해야 내려받을 수 있다.
1. https://huggingface.co 가입 (무료)
2. https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 에 들어가서 **"Agree and access repository"** 를 누른다. 페이지에 동의 버튼이 없으면 이미 열려 있는 것이다.
3. 오른쪽 위 프로필 → **Settings → Access Tokens → Create new token** → 종류 **Read** → 만들어진 `hf_...` 값을 복사
4. Colab 왼쪽 **열쇠 아이콘(보안 비밀)** → **새 보안 비밀 추가** → 이름 `HF_TOKEN`, 값에 토큰 붙여 넣기 → **노트북 액세스** 켜기

토큰은 비밀번호와 같다. 노트북 셀이나 git에 직접 적지 않는다.

## 1. 설치와 GPU 확인

- `bitsandbytes`: 모델을 4비트로 줄여 T4(메모리 15GB)에 올리게 해 준다. 원래 크기(약 14.5GB)로는 T4에서 메모리가 모자란다.
- `accelerate`: 모델을 GPU에 자동으로 올려 준다.
- `transformers`는 Colab에 이미 있지만 최신으로 올린다.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 T4 GPU로 바꿀 것")

## 2. 설정값과 표본 불러오기

- 표본은 GitHub 저장소의 `eot_pilot_sample.csv`를 바로 읽는다 (저장소가 공개라 로그인 필요 없음).
- 결과는 Google Drive에 저장한다. Colab은 오래 쓰지 않으면 연결이 끊기고, 끊기면 `/content` 안 파일이 사라지기 때문이다. Drive 연결 창이 뜨면 허용한다.
- `N_LIMIT`: 처음엔 **5**로 두고 끝까지 돌려서 출력이 제대로 나오는지 본다. 괜찮으면 `None`으로 바꿔 200건 전체를 돌린다.

In [ ]:
import json, re, time
from pathlib import Path
import pandas as pd
pd.set_option("display.max_colwidth", 200)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
GEN_CONFIG = dict(do_sample=True, temperature=0.2, top_p=0.95, top_k=25, max_new_tokens=2500)  # 논문 부록 B
SEED = 20261006
N_LIMIT = 5        # 시험 실행. 확인 후 None으로 바꿔 전체 실행

SAMPLE_URL = "https://raw.githubusercontent.com/kimjiwoo0603/Cart-To-Chart/main/data/processed/eot_pilot/eot_pilot_sample.csv"
sample = pd.read_csv(SAMPLE_URL, dtype={"review_id": str})
print("표본:", len(sample), "건", sample["period"].value_counts().to_dict())

from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/cart_to_chart/eot_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUT_DIR / "eot_raw.jsonl"
print("저장 위치:", OUT_DIR)

## 3. 모델 불러오기 (4비트)

- `HF_TOKEN`은 0단계에서 넣은 보안 비밀에서 읽는다.
- `nf4` 4비트 양자화: 무게(가중치)를 4비트로 압축해 메모리를 약 4분의 1로 줄인다. 계산은 16비트로 한다. 정확도가 조금 떨어질 수 있어서 **논문과 다른 점**으로 기록한다.
- 처음 실행 때 약 15GB를 내려받는다.

In [ ]:
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

hf_token = userdata.get("HF_TOKEN")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=hf_token, quantization_config=bnb, device_map="auto")
model.eval()
print("불러옴. GPU 메모리 사용:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

## 4. EOT-DETECT 프롬프트 (논문 4.3절 재구성)

논문은 프롬프트를 네 부분 P = ⟨S, T, I, R⟩ 로 정의하고 각 부분의 내용을 설명했지만, 문장 전체는 공개하지 않았다. 그래서 **논문이 설명한 요소를 빠짐없이, 논문과 부록 D의 표현을 최대한 그대로** 써서 다시 만들었다.

| 부분 | 논문 설명 (p.5518) | 아래 프롬프트 |
|---|---|---|
| **S** 시스템 메시지 | 모델의 전문가 역할과 전체 맥락 | 감정 분석·소비자 심리 전문가 (부록 D "Expert Judgment" 표현) |
| **T** 과제 설명 | 과제 정의 / Opinion Trigger 정의 / 감정 범위 / 지침 | 네 소제목으로 그대로 나눔. trigger 정의는 부록 D.3, 지침은 D.5·D.6과 경계 사례 |
| **I** 지시 5단계 | I1 핵심 감정 찾기 → I2 감정과 trigger 연결 → I3 균형 유지 → I4 명확하게 정리 → I5 최종 자기 점검 4가지 | 이름과 내용 그대로 |
| **R** 리뷰 | 분석할 리뷰 | review_id와 본문 |
| 출력 | 부록 D.8 JSON | JSON만 출력하라고 지시 |

**참고**: Mistral-7B-Instruct-v0.2는 대화 형식에 '시스템' 역할이 따로 없다. 그래서 S를 사용자 메시지 맨 앞에 붙인다 (이 모델에서 흔히 쓰는 방식).
**참고**: 팀 라벨링 지침의 [우리 추가] 판단 규칙(Joy vs Trust 등)은 **프롬프트에 넣지 않는다.** 논문 프롬프트를 그대로 재현해야 논문 결과(F1 0.86)와 비교할 수 있기 때문이다.

In [ ]:
S = """You are an expert in emotion analysis and consumer psychology, specializing in customer reviews from e-commerce platforms."""

T = """## Task Definition
Detect all emotions expressed in the customer review below and extract the Opinion Triggers for each emotion directly from the review text.

## Opinion Trigger Definition
An Opinion Trigger is a direct extract from the review that explains why the customer experienced a particular emotion. Each trigger must be:
- Extractive: an exact, contiguous substring of the review, exactly as it appears (same words, spelling, capitalization and punctuation).
- Clearly Linked: directly associated with the identified emotion.
- Self-Contained: understandable without needing additional context.

## Emotion Scope
Use only Plutchik's 8 primary emotions: Joy, Trust, Fear, Surprise, Sadness, Disgust, Anger, Anticipation.
If no clear emotion is expressed in the review, label it as Neutral with an empty trigger list.

## Guidelines
- Do not modify, paraphrase, summarize or correct the review text when extracting triggers.
- A review may express multiple emotions, and a single emotion may have multiple triggers.
- Annotate only emotions that are clearly expressed. Do not infer or speculate.
- Focus only on the presence or absence of an emotion, not its intensity.
- Use the entire review context to guide your decisions.
- Do not extract discontinuous spans; choose the most representative continuous span.
- When a shorter trigger is nested within a longer one, choose the span that is most informative and coherent.
- For comparative expressions, include the full comparison if it explains the emotion.
- In very short reviews with clear emotional signals, the entire review may serve as the trigger.
- In technical reviews, focus on emotional content, not technical details, unless a detail directly causes an emotional reaction.
- For sarcastic reviews, identify the intended emotion rather than the literal meaning."""

I = """## Instructions
Follow these steps:
1. Focus on Key Emotions: Read the review carefully and identify all relevant emotions. If none are found, label the review as Neutral.
2. Link Emotions to Opinion Triggers: For each emotion, extract one or more exact text spans from the review that explain why the emotion is present.
3. Maintain Balance: Capture both majority and minority emotions, so that no single emotion is over- or under-represented.
4. Keep it Clear: Finalize all detected emotions and their triggers, adhering strictly to the extractive requirement.
5. Final Self-Check: Before producing the final output, verify:
   (a) Emotion Coverage: all expressed emotions, including minority ones, are captured.
   (b) Opinion Trigger Coverage: all relevant triggers for each emotion are extracted.
   (c) Emotion Faithfulness: no extra, unwarranted emotions are added.
   (d) Opinion Trigger Verifiability: every trigger is an exact substring of the review.

## Output Format
Return only a JSON object in exactly this format, with no other text:
{"review_id": "<review_id>", "emotions": [{"emotion": "<one of the 8 emotions or Neutral>", "triggers": ["<exact span>", "..."]}]}"""

def build_prompt(review_id, text):
    R = f"""## Review
review_id: {review_id}
Review: {text}"""
    return "\n\n".join([S, T, I, R])

print(build_prompt("example123", "Only 143 sheets, not the 176 they claim. Very disappointed."))

## 5. [점검] 생성과 저장 (이어 돌리기 가능)

- 리뷰 하나씩 프롬프트를 넣고 논문 설정(temperature 0.2 등)으로 답을 생성한다.
- `set_seed(SEED)`: 리뷰마다 같은 시드를 다시 걸어, 같은 리뷰는 다시 돌려도 같은 답이 나오게 한다.
- 답은 **나오자마자 Drive의 `eot_raw.jsonl`에 한 줄씩 추가**한다. 연결이 끊기면 1~3단계를 다시 실행하고 이 셀을 실행하면, 이미 끝난 리뷰는 건너뛰고 이어서 한다.
- 처음엔 `N_LIMIT = 5`로 5건만 돌려 **출력 예시를 Claude에게 보낸다.** 형식이 괜찮으면 2단계에서 `N_LIMIT = None`으로 바꾸고 2단계와 이 셀만 다시 실행한다 (모델은 다시 안 불러와도 됨).

In [ ]:
done = set()
if RAW_PATH.exists():
    with open(RAW_PATH, encoding="utf-8") as f:
        done = {json.loads(line)["review_id"] for line in f if line.strip()}
todo = sample[~sample["review_id"].isin(done)]
if N_LIMIT is not None:
    todo = todo.head(max(0, N_LIMIT - len(done)))
print(f"이미 끝남 {len(done)}건 / 이번에 할 것 {len(todo)}건")

t0 = time.time()
for i, row in enumerate(todo.itertuples(), 1):
    messages = [{"role": "user", "content": build_prompt(row.review_id, row.text_clean)}]
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(model.device)
    set_seed(SEED)
    with torch.no_grad():
        out = model.generate(**enc, **GEN_CONFIG, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    with open(RAW_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"review_id": row.review_id, "output": answer}, ensure_ascii=False) + "\n")
    if i % 10 == 0 or i == len(todo):
        print(f"{i}/{len(todo)}건, {time.time() - t0:.0f}초")

raw = pd.read_json(RAW_PATH, lines=True, dtype={"review_id": str})
for r in raw.head(3).itertuples():
    print("\n[review]", sample.loc[sample.review_id == r.review_id, "text_clean"].iloc[0])
    print("[model]", r.output)

## 6. 출력 정리와 trigger 검증

모델 출력에서 JSON을 꺼내 표로 만든다.
1. **JSON 꺼내기**: 답의 첫 `{`부터 마지막 `}`까지를 JSON으로 읽는다. 앞뒤에 설명 문장이 붙어도 괜찮게 하기 위해서다. 시험 실행(5건)에서 모델이 JSON 문법을 틀리는 경우가 있어서 세 단계로 읽고, 어떤 방법으로 읽었는지 `parse_how`에 남긴다.
   - `json`: 그대로 읽힘
   - `repaired`: 작은따옴표 앞에 쓸데없는 `\`(`I\'m`)를 붙인 경우. 지우고 다시 읽음
   - `regex`: 괄호를 잘못 닫은 경우. `"emotion": ..., "triggers": [...]` 부분만 정규식으로 꺼냄
   - 셋 다 안 되면 `parse_ok = False`로 남기고 분석에서 뺀다. 내용은 고치지 않고 **형식만** 살리는 것이라, 꺼낸 trigger도 3단계 검증을 똑같이 거친다.
2. **감정 이름 정리**: 대소문자를 맞추고(`joy` → `Joy`), 9개 감정 밖의 이름은 버리고 개수를 센다.
3. **trigger 검증** (논문 3.3절 extractive constraint, I5-(d)):
   - `exact`: 리뷰 본문에 글자 그대로 있음 → **분석에 사용**
   - `loose`: 대소문자·공백·따옴표 모양만 다르면 찾을 수 있음 → 분석에서 빼지만 개수는 보고
   - `none`: 본문에 없음 (모델이 지어냄) → 제외
4. 감정만 있고 검증된 trigger가 하나도 없으면, 감정은 남기되 `has_valid_trigger = False`로 표시한다.

In [ ]:
EMOTIONS = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]
EMO_MAP = {e.lower(): e for e in EMOTIONS}

def _regex_rescue(s):
    """괄호가 틀린 JSON에서 "emotion": "...", "triggers": [...] 덩어리만 정규식으로 꺼낸다."""
    items = []
    blocks = re.split(r'\{\s*"emotion"\s*:', s)[1:]
    for b in blocks:
        m = re.match(r'\s*"([^"]*)"', b)
        if not m:
            continue
        t = re.search(r'"triggers"\s*:\s*\[(.*)', b, re.S)
        trigs = re.findall(r'"((?:[^"\\]|\\.)*)"', t.group(1)) if t else []
        trigs = [x.replace('\\"', '"').replace("\\'", "'") for x in trigs]
        items.append({"emotion": m.group(1), "triggers": trigs})
    return {"emotions": items} if items else None

def extract_json(s):
    """(결과, 방법)을 돌려준다. 방법: json(그대로 읽힘) / repaired(\\' 고친 뒤 읽힘) / regex(괄호 오류, 정규식으로 꺼냄) / None(실패)"""
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b <= a:
        return None, None
    body = s[a:b + 1]
    try:
        return json.loads(body), "json"
    except json.JSONDecodeError:
        pass
    try:
        return json.loads(body.replace("\\'", "'")), "repaired"
    except json.JSONDecodeError:
        pass
    obj = _regex_rescue(body)
    return (obj, "regex") if obj else (None, None)

def norm(s):
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", s).strip().lower()

def check_trigger(trig, text):
    if trig in text:
        return "exact"
    if norm(trig) and norm(trig) in norm(text):
        return "loose"
    return "none"

text_of = dict(zip(sample["review_id"], sample["text_clean"]))
trig_rows, emo_rows, bad_names = [], [], []
for r in raw.itertuples():
    obj, how = extract_json(r.output)
    text = text_of[r.review_id]
    if obj is None or not isinstance(obj.get("emotions"), list):
        emo_rows.append({"review_id": r.review_id, "parse_ok": False, "parse_how": None})
        continue
    found = {}
    for item in obj["emotions"]:
        if not isinstance(item, dict):
            continue
        name = EMO_MAP.get(str(item.get("emotion", "")).strip().lower())
        if name is None:
            bad_names.append(item.get("emotion"))
            continue
        trigs = [t for t in (item.get("triggers") or []) if isinstance(t, str) and t.strip()]
        for t in trigs:
            trig_rows.append({"review_id": r.review_id, "emotion": name, "trigger": t, "match": check_trigger(t, text)})
        found[name] = found.get(name, False) or any(check_trigger(t, text) == "exact" for t in trigs)
    row = {"review_id": r.review_id, "parse_ok": True, "parse_how": how}
    for e in EMOTIONS:
        row[e] = int(e in found)
    row["has_valid_trigger"] = all(found[e] for e in found if e != "Neutral") if found else False
    emo_rows.append(row)

triggers = pd.DataFrame(trig_rows, columns=["review_id", "emotion", "trigger", "match"])
emotions = pd.DataFrame(emo_rows)
emotions = sample[["review_id", "period"]].merge(emotions, on="review_id")
print("정리 끝:", len(emotions), "건 / 읽은 방법:", emotions["parse_how"].value_counts(dropna=False).to_dict())

## 7. [점검] 정리 결과 요약

- **JSON 읽기 성공률과 방법**: `regex`가 많으면 모델이 형식을 자주 틀린다는 뜻이라 보고서에 적는다.
- **9개 밖 감정 이름**: 모델이 지어낸 감정 이름 (예: Frustration). 몇 개인지 본다.
- **trigger 검증 비율**: `exact`가 높을수록 논문 조건을 잘 지킨 것이다. 논문은 이 비율을 따로 보고하지 않았지만, 검증 단계 자체(I5-(d))가 논문 방식이다.
- **구간별 감정 비율 (미리보기)**: 리뷰 중 그 감정이 나온 비율. 본격 비교는 사람 라벨로 모델을 검증한 뒤에 한다.

In [ ]:
print("JSON 읽기 성공:", int(emotions["parse_ok"].sum()), "/", len(emotions), "| 방법별:", emotions["parse_how"].value_counts().to_dict())
print("9개 밖 감정 이름:", len(bad_names), pd.Series(bad_names, dtype=object).value_counts().head(5).to_dict())
print("\ntrigger 검증:")
print(triggers["match"].value_counts().to_frame("개수").assign(비율=lambda d: (d["개수"] / d["개수"].sum()).round(3)))
ok = emotions[emotions["parse_ok"]]
print("\n구간별 감정 비율 (미리보기):")
ok.groupby("period")[EMOTIONS].mean().T.round(2)

In [ ]:
print("검증 안 된 trigger 예시 (loose / none):")
triggers[triggers["match"] != "exact"].head(10)

## 8. 저장

- `eot_triggers.csv`: 리뷰·감정·trigger 한 줄씩 (긴 형식). 검증 결과 `match` 포함
- `eot_emotions.csv`: 리뷰 한 줄에 감정 9개 0/1 (넓은 형식). 사람 라벨 시트와 같은 모양이라 비교하기 쉽다
- `eot_raw.jsonl`: 모델 원문 출력. 나중에 정리 규칙을 바꿔도 모델을 다시 돌릴 필요가 없게 그대로 둔다

전체(200건)를 돌린 뒤 Drive의 `MyDrive/cart_to_chart/eot_pilot/` 에서 세 파일을 내려받아 저장소 `data/processed/eot_pilot/`에 넣고 커밋한다.

In [ ]:
triggers.to_csv(OUT_DIR / "eot_triggers.csv", index=False, encoding="utf-8-sig")
emotions.to_csv(OUT_DIR / "eot_emotions.csv", index=False, encoding="utf-8-sig")
print("저장:", sorted(p.name for p in OUT_DIR.iterdir()))

## 9. 로그에 붙일 문장

아래 출력을 복사해 `logs/preprocessing_log.md` 맨 아래에 붙인다.

In [ ]:
mc = triggers["match"].value_counts()
lines = [
    "### 2026-10-06 T3. EOT-DETECT 감정·원인 구절 추출: 화장지 시험 (김지우)",
    "노트북: `notebooks/05_eot_detect_colab.ipynb` (Colab T4). 기준: Attri et al. (2025) 4.3절·부록 B·D",
    "",
    f"- 모델 {MODEL_ID}, 4비트(nf4) 양자화 (논문과 다른 점: T4 메모리 한계). 설정 temperature 0.2 / top_p 0.95 / top_k 25 / max_new_tokens 2500, 시드 {SEED}",
    "- 프롬프트: 논문이 전문을 공개하지 않아 S·T·I1~I5·R 구조와 부록 D 문구로 재구성. S는 사용자 메시지 앞에 붙임 (모델에 시스템 역할 없음)",
    f"- 리뷰 {len(emotions)}건 / JSON 읽기 성공 {int(emotions['parse_ok'].sum())}건 (방법별 {emotions['parse_how'].value_counts().to_dict()}) / 9개 밖 감정 이름 {len(bad_names)}개 {pd.Series(bad_names, dtype=object).value_counts().to_dict()}",
    f"- trigger {len(triggers)}개: 글자 그대로 {mc.get('exact', 0)} / 대소문자·공백만 다름 {mc.get('loose', 0)} / 본문에 없음 {mc.get('none', 0)} → 분석은 글자 그대로만 사용",
]
print("\n".join(lines))